# Results Analysis & Thesis Figures
### Master's Research: Machine Unlearning — Multi-Class Image Classification
**Author:** Mikołaj Hajder 264478

This notebook is **read-only with respect to training** — it only loads saved checkpoints / result JSONs and produces plots.

**Sections:**
1. Configuration
2. Imports
3. Load Checkpoints & Results JSON
4. Build Data Splits
5. Evaluation — All Splits
6. Training Curves
7. Per-Class Accuracy
8. SISA Specifics
9. ∇τ Specifics
10. Bar Chart — Accuracy Comparison
**11. MIA Score Comparison** ← NEW
**12. Loss & Entropy Distributions** ← NEW

> **Before running:** make sure you have run `runner_kaggle.ipynb` at least once
> so that checkpoints and result JSON files exist.


## 0. Configuration — set paths here
Paths are pre-configured for Kaggle. Uncomment the relevant block below if running locally or on Colab.


In [ ]:
import os, sys

# ── Kaggle paths ────────────────────────────────────────────────────
# CKPT_DIR  = '/kaggle/working/checkpoints'
# DATA_ROOT = '/kaggle/working/data'

# ── Local paths (uncomment when running locally) ────────────────────
CKPT_DIR  = '../checkpoints'
DATA_ROOT = '../data'

# ── Colab + Google Drive (uncomment when running on Colab) ──────────
# CKPT_DIR  = '/content/drive/MyDrive/master_thesis/checkpoints'
# DATA_ROOT = '/content/drive/MyDrive/master_thesis/data'

DATASET     = 'CIFAR10'       # 'CIFAR10' | 'CIFAR100'
NUM_CLASSES = 10 if DATASET == 'CIFAR10' else 100
ds_tag      = DATASET.lower()

# SISA results live in a sub-directory per dataset
SISA_DIR = os.path.join(CKPT_DIR, f'sisa_{ds_tag}')

# Add repo root to path (needed on Kaggle / Colab)
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(''), '..')))

print(f'Checkpoint dir : {os.path.abspath(CKPT_DIR)}')
print(f'SISA dir       : {os.path.abspath(SISA_DIR)}')
print(f'Dataset        : {DATASET}')


## 1. Imports

In [ ]:
import json
import sklearn
from mia import _compute_features, _train_attacker


## 2. Load Checkpoints

In [ ]:
original_ckpt      = os.path.join(CKPT_DIR, f'resnet18_{ds_tag}_best.pth')

# ── Naive results JSON (contains MIA scores) ──────────────────────
naive_results      = {}
naive_results_path = os.path.join(CKPT_DIR, f'naive_{ds_tag}_results.json')
if os.path.exists(naive_results_path):
    with open(naive_results_path) as f:
        naive_results = json.load(f)
    print(f'Loaded naive results JSON: {naive_results_path}')
else:
    print('No naive results JSON found (run unlearn_naive.py first).')


## 3. Build Data Splits

In [ ]:
import random, torchvision

full_train, test_ds = get_datasets(DATASET, DATA_ROOT)
DatasetClass = (torchvision.datasets.CIFAR10 if DATASET == 'CIFAR10'
                else torchvision.datasets.CIFAR100)
full_eval = DatasetClass(root=DATA_ROOT, train=True, download=True,
                         transform=get_test_transform(DATASET))

# Reconstruct forget / retain split using same seed as unlearning script
seed = naive_cfg.get('seed', 42)
rng  = random.Random(seed)
all_indices = list(range(len(full_train)))

if forget_strategy == 'random':
    frac = naive_cfg.get('forget_fraction', 0.01)
    forget_indices = rng.sample(all_indices, int(len(all_indices) * frac))
elif forget_strategy == 'class':
    cls = naive_cfg.get('forget_class', 0)
    forget_indices = [i for i, (_, label) in enumerate(full_train) if label == cls]
else:
    forget_indices = []

forget_set     = set(forget_indices)
retain_indices = [i for i in all_indices if i not in forget_set]

bs = naive_cfg.get('batch_size', 256)
retain_eval_loader = DataLoader(Subset(full_eval, retain_indices),
                                batch_size=bs, shuffle=False, num_workers=2)
forget_loader      = DataLoader(Subset(full_eval, forget_indices),
                                batch_size=bs, shuffle=False, num_workers=2)
test_loader        = DataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=2)

print(f'Forget : {len(forget_indices):,}  |  Retain : {len(retain_indices):,}  |  Test : {len(test_ds):,}')


## 4. Evaluation — All Splits

In [ ]:
criterion = nn.CrossEntropyLoss()

# ── Original ─────────────────────────────────────────────────────────
orig_retain_loss, orig_retain_acc = evaluate(original_model, retain_eval_loader, criterion, device)
orig_forget_loss, orig_forget_acc = evaluate(original_model, forget_loader,      criterion, device)
orig_test_loss,   orig_test_acc   = evaluate(original_model, test_loader,        criterion, device)

print(f"{'='*62}")
print(f"{'Metric':<20} {'Original':>12}")
print(f"{'='*62}")
print(f"{'Retain Accuracy':<20} {orig_retain_acc:>11.2f}%")
print(f"{'Forget Accuracy':<20} {orig_forget_acc:>11.2f}%")
print(f"{'Test Accuracy':<20} {orig_test_acc:>11.2f}%")

# ── Naive ─────────────────────────────────────────────────────────
naive_retain_acc = naive_forget_acc = naive_test_acc = float('nan')
if naive_model is not None:
    naive_retain_loss, naive_retain_acc = evaluate(naive_model, retain_eval_loader, criterion, device)
    naive_forget_loss, naive_forget_acc = evaluate(naive_model, forget_loader,      criterion, device)
    naive_test_loss,   naive_test_acc   = evaluate(naive_model, test_loader,        criterion, device)

# ── SISA ─────────────────────────────────────────────────────────
sisa_after      = sisa_results.get('after', {})
sisa_retain_acc = sisa_after.get('retain_acc', float('nan'))
sisa_forget_acc = sisa_after.get('forget_acc', float('nan'))
sisa_test_acc   = sisa_after.get('test_acc',   float('nan'))
sisa_time       = sisa_results.get('unlearn_time_s', float('nan'))
sisa_train_time = sisa_results.get('sisa_train_time_s', 0)
have_sisa       = bool(sisa_results)

# ── ∇τ ─────────────────────────────────────────────────────────
gt_after       = grad_tau_results.get('after', {})
gt_retain_acc  = gt_after.get('retain_acc', float('nan'))
gt_forget_acc  = gt_after.get('forget_acc', float('nan'))
gt_test_acc    = gt_after.get('test_acc',   float('nan'))
gt_time        = grad_tau_results.get('unlearn_time_s', float('nan'))
have_gt        = bool(grad_tau_results)

# ── Accuracy comparison table ───────────────────────────────────────
if naive_model is not None:
    header_parts = [f"{'Metric':<20}", f"{'Original':>12}", f"{'Naive Retrain':>14}"]
    if have_sisa: header_parts.append(f"{'SISA Unlearn':>14}")
    if have_gt:   header_parts.append(f"{∇τ Unlearn':>14}")
    header_parts += [f"{Δ Naive':>9}"]
    if have_sisa: header_parts.append(f"{Δ SISA':>9}")
    if have_gt:   header_parts.append(f"{Δ ∇τ':>9}")

    header = '  '.join(header_parts)
    print(f"\n{'='*len(header)}")
    print(header)
    print(f"{'='*len(header)}")

    rows = [
        ('Retain Accuracy', orig_retain_acc, naive_retain_acc, sisa_retain_acc, gt_retain_acc),
        ('Forget Accuracy', orig_forget_acc, naive_forget_acc, sisa_forget_acc, gt_forget_acc),
        ('Test Accuracy',   orig_test_acc,   naive_test_acc,   sisa_test_acc,   gt_test_acc),
    ]
    for name, orig, naive, sisa, gt in rows:
        d_naive = naive - orig
        d_sisa  = sisa  - orig
        d_gt    = gt    - orig
        s_naive = '+' if d_naive >= 0 else ''
        s_sisa  = '+' if d_sisa  >= 0 else ''
        s_gt    = '+' if d_gt    >= 0 else ''
        row = f"{name:<20}  {orig:>11.2f}%  {naive:>13.2f}%"
        if have_sisa: row += f"  {sisa:>13.2f}%"
        if have_gt:   row += f"  {gt:>13.2f}%"
        row += f"  {s_naive}{d_naive:>8.2f}%"
        if have_sisa: row += f"  {s_sisa}{d_sisa:>8.2f}%"
        if have_gt:   row += f"  {s_gt}{d_gt:>8.2f}%"
        print(row)
    print(f"{'='*len(header)}")

    # ── Timing table ──────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"{'Method':<30} {'Train Time':>18} {'Unlearn Time':>18}")
    print(f"{'='*70}")
    print(f"{'Original / Naive Retrain':<30} {orig_time:>17.1f}s {naive_time:>17.1f}s")
    if have_sisa:
        print(f"{'SISA Ensemble':<30} {sisa_train_time:>17.1f}s {sisa_time:>17.1f}s")
    if have_gt:
        print(f"{'∇τ Approx. Unlearn':<30} {'n/a':>17}  {gt_time:>17.1f}s")
    print(f"{'='*70}")
    if naive_time > 0:
        if have_sisa and sisa_time > 0:
            print(f"Unlearning speedup (SISA vs Naive) : {naive_time/sisa_time:.1f}x")
        if have_gt and gt_time > 0:
            print(f"Unlearning speedup (∇τ vs Naive)  : {naive_time/gt_time:.1f}x")


## 5. Training Curves

In [ ]:
def plot_curves(history, title, best_acc=None):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs, history['train_loss'], label='Train', linewidth=2)
    axes[0].plot(epochs, history['test_loss'],  label='Test',  linewidth=2)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].set_title(f'{title} — Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    best_label = f'Test (best: {best_acc:.2f}%)' if best_acc else 'Test'
    axes[1].plot(epochs, history['train_acc'], label='Train',      linewidth=2)
    axes[1].plot(epochs, history['test_acc'],  label=best_label,   linewidth=2)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title(f'{title} — Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    return fig

if orig_history:
    fig = plot_curves(orig_history, f'{DATASET} — Original Training',
                      best_acc=orig_ckpt_data.get('test_acc'))
    fig.savefig(os.path.join(CKPT_DIR, f'resnet18_{ds_tag}_training_curves.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No training history found in original checkpoint.')


In [ ]:
if naive_history:
    fig = plot_curves(naive_history,
                      f'{DATASET} — Naive Unlearning ({forget_strategy}, {forget_size} samples)',
                      best_acc=naive_ckpt_data.get('test_acc'))
    fig.savefig(os.path.join(CKPT_DIR, f'resnet18_{ds_tag}_naive_unlearn_curves.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No training history found in naive checkpoint.')


## 6. Per-Class Accuracy

In [ ]:
orig_per_class  = per_class_accuracy(original_model, test_loader, NUM_CLASSES, device)

fig, ax = plt.subplots(figsize=(12, 4))
x = range(NUM_CLASSES)
ax.bar([i - 0.2 for i in x], orig_per_class, width=0.4, label='Original', alpha=0.8)

if naive_model is not None:
    naive_per_class = per_class_accuracy(naive_model, test_loader, NUM_CLASSES, device)
    ax.bar([i + 0.2 for i in x], naive_per_class, width=0.4, label='Naive Unlearn', alpha=0.8)

ax.set_xlabel('Class'); ax.set_ylabel('Accuracy (%)')
ax.set_title(f'{DATASET} — Per-Class Test Accuracy')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(CKPT_DIR, f'resnet18_{ds_tag}_per_class_acc.png'),
            dpi=150, bbox_inches='tight')
plt.show()


## 7. SISA Specifics
Detailed view of the SISA ensemble structure and unlearning performance.
Only populated when `unlearn_sisa.py` has been run and its results loaded in Section 2.


In [ ]:
if sisa_results:
    print(f"Dataset          : {sisa_results['dataset']}")
    print(f"Shards / Slices  : {sisa_results['num_shards']} / {sisa_results['num_slices']}")
    print(f"Aggregation      : {sisa_results['aggregation']}")
    print(f"Affected Shards  : {sisa_results['affected_shards']}")
    print(f"Unlearning Time  : {sisa_results['unlearn_time_s']:.2f}s")

    before = sisa_results.get('before', {})
    after  = sisa_results.get('after',  {})
    print(f"\n{'='*65}")
    print(f"{'Metric':<22} {'Before':>12}  {'After':>12}  {'Δ':>8}")
    print(f"{'='*65}")
    for key, label in [('retain_acc','Retain Acc'),('forget_acc','Forget Acc'),('test_acc','Test Acc')]:
        b = before.get(key, float('nan'))
        a = after.get(key, float('nan'))
        d = a - b; s = '+' if d >= 0 else ''
        print(f"{label:<22} {b:>11.2f}%  {a:>11.2f}%  {s}{d:>7.2f}%")
    print("-" * 65)
    for key, label in [('mia_l','MIA_L (loss)'),('mia_e','MIA_E (entropy)')]:
        b = before.get(key, float('nan'))
        a = after.get(key, float('nan'))
        d = a - b; s = '+' if d >= 0 else ''
        print(f"{label:<22} {b*100:>11.2f}%  {a*100:>11.2f}%  {s}{d*100:>7.2f}%   (ideal 50%)")
    print(f"{'='*65}")

    print("\nShard Stats:")
    print(f"{'Shard':<8} {'Affected':>10} {'Slices Retrained':>18} {'Time (s)':>10}")
    print("-" * 50)
    for shard in sisa_results.get('shard_stats', []):
        affected = 'Yes' if shard['retrained'] else 'No'
        print(f"{shard['shard_id']:<8} {affected:>10} {shard.get('slices_retrained',0):>18} {shard.get('retrain_time_s',0):>10.1f}")
else:
    print('SISA results not available. Run 5g–5i in runner_kaggle.ipynb first.')


## 8. ∇τ Specifics
Detailed view of the gradient-based approximate unlearning run.
Only populated when `unlearn_grad_tau.py` has been run (cells 5j–5l in runner_kaggle.ipynb).


In [ ]:
if grad_tau_results:
    print(f"Dataset          : {grad_tau_results['dataset']}")
    print(f"Method           : {grad_tau_results['method']}")
    print(f"Forget strategy  : {grad_tau_results['forget_strategy']}  ({grad_tau_results['forget_size']} samples)")
    print(f"Alpha init (α₀)  : {grad_tau_results.get('alpha_init', 'n/a')}")
    print(f"Forget epochs    : {grad_tau_results.get('forget_epochs', 'n/a')}")
    print(f"Unlearning time  : {grad_tau_results.get('unlearn_time_s', 0):.2f}s")

    before = grad_tau_results.get('before', {})
    after  = grad_tau_results.get('after',  {})
    print(f"\n{'='*65}")
    print(f"{'Metric':<22} {'Before':>12}  {'After':>12}  {'Δ':>8}")
    print(f"{'='*65}")
    for key, label in [('retain_acc','Retain Acc'),('forget_acc','Forget Acc'),('test_acc','Test Acc')]:
        b = before.get(key, float('nan'))
        a = after.get(key, float('nan'))
        d = a - b; s = '+' if d >= 0 else ''
        print(f"{label:<22} {b:>11.2f}%  {a:>11.2f}%  {s}{d:>7.2f}%")
    print("-" * 65)
    for key, label in [('mia_l','MIA_L (loss)'),('mia_e','MIA_E (entropy)')]:
        b = before.get(key, float('nan'))
        a = after.get(key, float('nan'))
        d = a - b; s = '+' if d >= 0 else ''
        print(f"{label:<22} {b*100:>11.2f}%  {a*100:>11.2f}%  {s}{d*100:>7.2f}%   (ideal 50%)")
    print(f"{'='*65}")
else:
    print('∇τ results not available. Run cells 5j–5l in runner_kaggle.ipynb first.')


## 9. Bar Chart — Retain / Forget / Test Accuracy Comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

methods  = ['Original']
retain_v = [orig_retain_acc]
forget_v = [orig_forget_acc]
test_v   = [orig_test_acc]

if naive_model is not None:
    methods.append('Naive Retrain')
    retain_v.append(naive_retain_acc)
    forget_v.append(naive_forget_acc)
    test_v.append(naive_test_acc)

if sisa_results:
    after = sisa_results.get('after', {})
    methods.append('SISA Unlearn')
    retain_v.append(after.get('retain_acc', 0))
    forget_v.append(after.get('forget_acc', 0))
    test_v.append(after.get('test_acc', 0))

if grad_tau_results:
    gt_after = grad_tau_results.get('after', {})
    methods.append('∇τ Unlearn')
    retain_v.append(gt_after.get('retain_acc', 0))
    forget_v.append(gt_after.get('forget_acc', 0))
    test_v.append(gt_after.get('test_acc', 0))

x     = np.arange(len(methods))
width = 0.25
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width, retain_v, width, label='Retain Acc', alpha=0.85)
ax.bar(x,         forget_v, width, label='Forget Acc', alpha=0.85)
ax.bar(x + width, test_v,   width, label='Test Acc',   alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(methods)
ax.set_ylabel('Accuracy (%)')
ax.set_title(f'{DATASET} — Unlearning Method Comparison')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(CKPT_DIR, f'resnet18_{ds_tag}_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()


## 10. MIA Score Comparison

Reads `mia_l` / `mia_e` from all result JSON files.  
**Ideal post-unlearning score = 50%** (attacker cannot distinguish forget set from test set).  
Score > 50% → model still remembers the forget set.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Collect MIA scores from all result JSONs ──────────────────────────────
mia_rows = []  # (method, mia_l_before, mia_l_after, mia_e_before, mia_e_after)

def _mia_from(results, label):
    if not results: return
    b = results.get('before', {}); a = results.get('after', {})
    mia_rows.append((label,
                     b.get('mia_l', float('nan')), a.get('mia_l', float('nan')),
                     b.get('mia_e', float('nan')), a.get('mia_e', float('nan'))))

_mia_from(naive_results,     'Naive Retrain')
_mia_from(sisa_results,      'SISA')
_mia_from(grad_tau_results,  '∇τ')

# ── Print table ───────────────────────────────────────────────────────────
print(f"{'Method':<18} {'MIA_L Before':>13} {'MIA_L After':>12} {'MIA_E Before':>13} {'MIA_E After':>12}  Note")
print('-' * 80)
for row in mia_rows:
    label, lb, la, eb, ea = row
    tag = '✓ good' if (abs(la-0.5)<0.05 and abs(ea-0.5)<0.05) else ''
    print(f"{label:<18} {lb*100:>12.2f}% {la*100:>11.2f}% {eb*100:>12.2f}% {ea*100:>11.2f}%  {tag}")
print(f"\n{'Ideal (retrain baseline)':<18} {'—':>13} {'50.00%':>12} {'—':>13} {'50.00%':>12}")

if not mia_rows:
    print('No MIA results found. Run unlearning scripts first.')


In [ ]:
if mia_rows:
    labels   = [r[0] for r in mia_rows]
    mia_l_b  = [r[1]*100 for r in mia_rows]
    mia_l_a  = [r[2]*100 for r in mia_rows]
    mia_e_b  = [r[3]*100 for r in mia_rows]
    mia_e_a  = [r[4]*100 for r in mia_rows]

    x = np.arange(len(labels)); w = 0.20
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    for ax, before, after, title in [
            (axes[0], mia_l_b, mia_l_a, 'MIA_L  (loss-based)'),
            (axes[1], mia_e_b, mia_e_a, 'MIA_E  (entropy-based)')]:
        ax.bar(x - w/2, before, w, label='Before unlearning', alpha=0.75, color='steelblue')
        ax.bar(x + w/2, after,  w, label='After unlearning',  alpha=0.85, color='tomato')
        ax.axhline(50, color='green', linestyle='--', linewidth=1.5, label='Ideal (50%)')
        ax.set_xticks(x); ax.set_xticklabels(labels, rotation=10)
        ax.set_ylabel('Attacker Accuracy (%)'); ax.set_ylim(30, 100)
        ax.set_title(f'{DATASET} — {title}'); ax.legend(); ax.grid(axis='y', alpha=0.3)

    plt.suptitle('Membership Inference Attack — Before vs After Unlearning', fontsize=13, y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(CKPT_DIR, f'mia_comparison_{ds_tag}.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {CKPT_DIR}/mia_comparison_{ds_tag}.png')


## 11. Loss & Entropy Distributions

Visualises the per-sample loss and entropy distributions for the **forget set** vs the **test set** before and after unlearning.  
After perfect unlearning the two distributions should overlap completely.


In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def get_features(model, loader, device):
    model.eval()
    crit = torch.nn.CrossEntropyLoss(reduction='none')
    losses, entropies = [], []
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        logits = model(imgs)
        losses.append(crit(logits, lbls).cpu().numpy())
        probs = F.softmax(logits, dim=1)
        entropies.append((-(probs * torch.log(probs + 1e-12)).sum(1)).cpu().numpy())
    import numpy as np
    return np.concatenate(losses), np.concatenate(entropies)

# Collect (model_label, model_object) pairs
model_pairs = [('Original', original_model)]
if naive_model is not None:
    model_pairs.append(('Naive Retrain', naive_model))

# Load grad_tau checkpoint if available
gt_ckpt_path = os.path.join(CKPT_DIR, f'resnet18_{ds_tag}_grad_tau_unlearn.pth')
if os.path.exists(gt_ckpt_path):
    from utils import load_checkpoint
    gt_model = load_checkpoint(gt_ckpt_path, device)
    model_pairs.append(('∇τ Unlearn', gt_model))

n_models = len(model_pairs)
fig, axes = plt.subplots(n_models, 2, figsize=(14, 4 * n_models))
if n_models == 1: axes = axes[None, :]

for row, (label, mdl) in enumerate(model_pairs):
    f_loss, f_ent = get_features(mdl, forget_loader, device)
    t_loss, t_ent = get_features(mdl, test_loader,   device)

    for col, (f_feat, t_feat, feat_name) in enumerate([
            (f_loss, t_loss, 'Cross-Entropy Loss'),
            (f_ent,  t_ent,  'Output Entropy')]):
        ax = axes[row, col]
        lo = min(f_feat.min(), t_feat.min())
        hi = max(f_feat.max(), t_feat.max())
        bins = np.linspace(lo, hi, 60)
        ax.hist(f_feat, bins=bins, alpha=0.6, label='Forget set', density=True, color='tomato')
        ax.hist(t_feat, bins=bins, alpha=0.6, label='Test set',   density=True, color='steelblue')
        ax.set_xlabel(feat_name); ax.set_ylabel('Density')
        ax.set_title(f'{label} — {feat_name}')
        ax.legend(); ax.grid(alpha=0.3)

plt.suptitle(f'{DATASET} — Feature Distributions (Forget vs Test)', fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(CKPT_DIR, f'mia_distributions_{ds_tag}.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {CKPT_DIR}/mia_distributions_{ds_tag}.png')
